# SATNAC 2026 : From Action Recognition to Sign Language Recognition: Scaling to Large Vocabularies Under Resource Constraints


## Specifications: 

All models run on all four WLASL splits (preprocessed to remove videos with length <= 9)

### In the currently submitted draft:

As the paper stands, this is the current spec, one of the following variations may be useful later.

| Models            | Patience  | Number of frames  | 
|-------------------|-----------|-------------------|
| MViTv2_S, S3D     | 50        | 16                |

### Varations: 

Essentially two extensions in various directions, one complete the other would need further runs. 

#### All Models:

PyTorch models were only run at 16 frames (3D CNNs would need to at least be run at 32 to be fair). These same models were only run at a patience of 50.

| Models            | Patience  | Number of frames  | 
|-------------------|-----------|-------------------|
| All PyTorch       | 50        | 16                |

#### Aso at 32 frames:

Technicall MViTv2_B_32x3 was aslo run at 32 frames, but it wasn't run at 16 frames, has a different size to MViTv2_S, and is a slowfast model, so might as well be left out. These experiments used a patience of 15.


| Models            | Patience  | Number of frames  | 
|-------------------|-----------|-------------------|
| MViTv2_S(_e), S3D | 15        | 16, 32            |


In [32]:
import json
from pathlib import Path

#locals
import pandas as pd

# from src.results import fetch_runs
from src.results import get_filters_drop_keys, search_old_runs, unpack_filters
from src.run_types import CUTOFF_9_NAMES, RESULTS_DIR, RESULTS_OUTPUTS


In [33]:
project_name = 'satnac_2026'
results_dir = RESULTS_DIR / project_name
output_dir = RESULTS_OUTPUTS / project_name
# runs = fetch_runs(results_dir / 'filters.py')

#Currently published:
target_lengths = [16]
patiences = [50]
models = ['S3D', 'MViTv2_S'] 

#Also at 32 frames:
# target_lengths = [16, 32]
# patiences = [15]
# models = ['S3D', 'MViTv2_S', 'MViTv2_S_e'] 


file_filters, _ = get_filters_drop_keys(results_dir / 'filters.py')

additional_filters = {
    'data' : {'target_length' : lambda x: x in target_lengths},
    'stopping': {'patience' : lambda x: x in patiences},
    'admin': {
        'config_path': lambda x: 'debug' not in x,
        'model': lambda x: x in models
              }
}

file_filter_keys, file_crits = unpack_filters(file_filters)
add_filter_keys, add_crits = unpack_filters(additional_filters)


runs = search_old_runs(
    filter_key_sets=file_filter_keys + add_filter_keys,
    criterions=file_crits + add_crits,
    sort_keys=[['results', 'test', 'average_loss']]
)

print(f'{len(runs)} found')



10 found


## Now we can compare the runs

### Load into data frame, including number of frames and early stopping patience

In [34]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

##### Create DF

In [35]:
df_format = []

for run in runs:
    df_format.append(
        {
            "model": run.admin.model,
            "exp_no": run.admin.exp_no,
            "run_id": run.wandb.run_id,
            "split": run.admin.split,
            "patience": run.stopping.patience if run.stopping else None,
            "No. frames": run.data.target_length
        }
        
        | {k: v for k, v in run.results.model_dump()[set_name][acc_type].items()}
        | {"config path": run.admin.config_path,
            "weight path": run.admin.weight_path}
    )    

df = pd.DataFrame(df_format) 

df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})

df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')


##### Display DF

In [36]:

splits = CUTOFF_9_NAMES
for split_name in splits:
    print(f'{split_name}'.capitalize())
    subdf = df[df['split'] == split_name]
    for patience in patiences:
        print(f'Patience: {patience}')
        pdf = subdf[subdf['patience'] == patience]
        for tl in target_lengths:
            print(f'No. Frames: {tl}')
            tdf = pdf[pdf['No. frames'] == tl]
            display(tdf.sort_values('Top-1', ascending=False))
    
    # display(subdf.sort_values('Top-1', ascending=False))

Asl100_cutoff_9
Patience: 50
No. Frames: 16


,model,exp_no,run_id,split,patience,No. frames,Top-1,Top-5,Top-10,config path,weight path
0,MViTv2_S,007,None,asl100_cutoff_9,50,16,79.46,91.86,95.35,configfiles/generic/lframe_hwd_warmrestarts.toml,None
1,MViTv2_S,010,None,asl100_cutoff_9,50,16,73.26,91.47,94.57,/home/luke/Code/SLR/src/configfiles/asl100/MVi...,None
5,S3D,051,f6r2dos3,asl100_cutoff_9,50,16,53.10,79.84,87.98,configfiles/asl100/S3D/exp046.toml,None
4,S3D,046,5xgru0cs,asl100_cutoff_9,50,16,50.39,77.91,86.43,configfiles/asl100/S3D/exp046.toml,None


Asl300_cutoff_9
Patience: 50
No. Frames: 16


,model,exp_no,run_id,split,patience,No. frames,Top-1,Top-5,Top-10,config path,weight path
2,MViTv2_S,001,nfwehytd,asl300_cutoff_9,50,16,65.42,88.92,93.56,configfiles/asl300/MViTv2_S/exp001.toml,None
6,S3D,002,vty80mdi,asl300_cutoff_9,50,16,51.95,78.44,87.13,configfiles/asl300/S3D/exp002.toml,None


Asl1000_cutoff_9
Patience: 50
No. Frames: 16


,model,exp_no,run_id,split,patience,No. frames,Top-1,Top-5,Top-10,config path,weight path
3,MViTv2_S,000,rc5m3meh,asl1000_cutoff_9,50,16,53.41,81.77,87.79,configfiles/asl1000/MViTv2_S/exp000.toml,None
7,S3D,000,kpdu2hra,asl1000_cutoff_9,50,16,36.25,67.80,77.08,configfiles/asl1000/S3D/exp000.toml,None


Asl2000_cutoff_9
Patience: 50
No. Frames: 16


,model,exp_no,run_id,split,patience,No. frames,Top-1,Top-5,Top-10,config path,weight path
8,MViTv2_S,000,fkv6kpik,asl2000_cutoff_9,50,16,38.87,71.62,79.82,configfiles/asl2000/MViTv2_S/exp000.toml,None
9,S3D,000,olp97b32,asl2000_cutoff_9,50,16,19.62,46.96,58.25,configfiles/asl2000/S3D/exp000.toml,None


### To LaTeX


In [37]:
# Select and rename the columns needed for the thesis table
from src.visualise2 import split_name_mapper

cols = {
        "model": "Model",
        "No. frames": "Frames",
        "Top-1": "Acc@1",
        "Top-5": "Acc@5",
        "Top-10": "Acc@10",
    }

latex_names = {
    'MViTv2_S_e' : 'MViTv2-S$^I$',
    'MViTv2_S' : 'MViTv2-S',
    'S3D' : 'S3D'
}

for split in CUTOFF_9_NAMES:
    
    subdf = df[df['split'] == split]
    tex_df = subdf[list(cols.keys())].rename(columns=cols)

    # Sort by loss (ascending — best/lowest loss first)
    tex_df = tex_df.sort_values("Acc@1").reset_index(drop=True)
    
    # Export as a LaTeX table (rough draft — formatting/labels will still need cleanup)
    latex_str = tex_df.to_latex(
        index=False,
        float_format="%.2f",
        column_format="lcccc",
        caption=f"Top-k accuracy on {split_name_mapper(split)} at 16 and 32 frames",
        label=f"tab:res_{project_name}",
    )

    for k, v in latex_names.items():
        latex_str = latex_str.replace(k, v)

    output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_dir / f"{project_name}_{split}_results.tex", "w") as f:
        f.write(latex_str)

    # print(tex_df.to_string(index=False))
    print(f"\n--- LaTeX : {split} ---\n")
    print(latex_str)


--- LaTeX : asl100_cutoff_9 ---

\begin{table}
\caption{Top-k accuracy on WLASL-100 at 16 and 32 frames}
\label{tab:res_satnac_2026}
\begin{tabular}{lcccc}
\toprule
Model & Frames & Acc@1 & Acc@5 & Acc@10 \\
\midrule
S3D & 16 & 50.39 & 77.91 & 86.43 \\
S3D & 16 & 53.10 & 79.84 & 87.98 \\
MViTv2-S & 16 & 73.26 & 91.47 & 94.57 \\
MViTv2-S & 16 & 79.46 & 91.86 & 95.35 \\
\bottomrule
\end{tabular}
\end{table}


--- LaTeX : asl300_cutoff_9 ---

\begin{table}
\caption{Top-k accuracy on WLASL-300 at 16 and 32 frames}
\label{tab:res_satnac_2026}
\begin{tabular}{lcccc}
\toprule
Model & Frames & Acc@1 & Acc@5 & Acc@10 \\
\midrule
S3D & 16 & 51.95 & 78.44 & 87.13 \\
MViTv2-S & 16 & 65.42 & 88.92 & 93.56 \\
\bottomrule
\end{tabular}
\end{table}


--- LaTeX : asl1000_cutoff_9 ---

\begin{table}
\caption{Top-k accuracy on WLASL-1000 at 16 and 32 frames}
\label{tab:res_satnac_2026}
\begin{tabular}{lcccc}
\toprule
Model & Frames & Acc@1 & Acc@5 & Acc@10 \\
\midrule
S3D & 16 & 36.25 & 67.80 & 77.08 \\

# Continue in the Next Notebook:

[correlation_f1.ipynb](./correlation_f1.ipynb)